# Variants in DNAJC13 and their Association with Parkinson's Disease Across Different Ancestral Backgrounds

* **Project**: gp2-hackathon-2023/Team 5 DNAJC13 Gene Assesment
* **Last updated**: Dec-2025
* **Version**: Python3.9
* **Data**: GP2 Release 11

# Summary
This notebook is focused on analysing association between DNAJC13 variants in all ancestries and Parkinson's disease.

# Imports

In [1]:
# Use the os package to interact with the environment
import os
import sys

# Bring in Pandas for Dataframe functionality
import pandas as pd
from functools import reduce

# Bring some visualization functionality
import seaborn as sns

# numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Enable interaction with the FireCloud API
# from firecloud import api as fapi

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

/tmp/ipykernel_676/627905169.py:26: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Set base directory

In [2]:
# Define chromosome of interest
CHNUM = 3
debug = 0
# Define the root directory to store the analysis results
BASE_DIR = "/home/jupyter/workspace/ws_files/202512_R11_Batch"
os.makedirs(BASE_DIR, exist_ok=True) # Create folder if it doesn't exist

# Define the ancestries to be analyzed

if (debug):
    ancestries = {'AAC'} #For debugging we use only one ancestry
    print(f'Debug mode on')
else:
    ancestries = {'AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS'}
    print(f'Debug mode off')

Debug mode off


# Install Packages

## PLINK

In [ ]:
%%bash

mkdir -p /home/jupyter/tools
cd /home/jupyter/tools

if test -e /home/jupyter/tools/plink; then
echo "Plink1.9 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink \n    -------"
wget -N http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip
unzip -o plink_linux_x86_64_20190304.zip
echo -e "\n plink downloaded and unzipped in /home/jupyter/tools \n "

fi

In [ ]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink2; then
echo "Plink2 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink2 \n    -------"
# wget -N https://s3.amazonaws.com/plink2-assets/plink2_linux_amd_avx2_20240704.zip
wget -N https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_amd_avx2_20250129.zip
unzip -o plink2_linux_amd_avx2_20250129.zip
echo -e "\n plink2 downloaded and unzipped in /home/jupyter/tools \n "
fi

## Bgzip and Tabix

In [ ]:
## Install bgzip

! pip install --user bgzip

In [ ]:
## Install tabix

! conda install bioconda::tabix -y

## ANNOVAR

In [ ]:
%%bash

# Install ANNOVAR:
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/tools/annovar; then

echo "annovar is already installed in /home/jupyter/tools/"
else
echo "annovar is not installed"
cd /home/jupyter/tools/

wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

tar xvfz annovar.latest.tar.gz

fi

In [ ]:
%%bash

# Install ANNOVAR: Download resources for annotation

cd /home/jupyter/tools/annovar/

perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20140902 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb cytoBand humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ensGene humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar exac03 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar avsnp147 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp30a humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad211_genome humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ljb26_all humandb/

## RVTests

In [ ]:
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /home/jupyter/tools/rvtests; then

echo "rvtests is already installed"
else
echo "rvtests is not installed"

mkdir /home/jupyter/tools/rvtests
cd /home/jupyter/tools/rvtests

wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz

tar -zxvf rvtests_linux64.tar.gz
fi

In [ ]:
# chmod to make sure you have permission to run the program
! chmod u+x /home/jupyter/tools/plink
! chmod u+x /home/jupyter/tools/plink2
! chmod 777 /home/jupyter/tools/rvtests/executable/rvtest

# Reference Files

In [3]:
GP2_CLINICAL_RELEASE_PATH = "/home/jupyter/workspace/gp2_tier2_eu_release11"
MASTER_KEY = f"{GP2_CLINICAL_RELEASE_PATH}/clinical_data/master_key_release11_final_vwb.csv"

GP2_META_RELEASE_PATH = f"{GP2_CLINICAL_RELEASE_PATH}/meta_data"

GP2_RAW_GENO_PATH = f"{GP2_CLINICAL_RELEASE_PATH}/raw_genotypes"

#GP2_RAW_FILENAME = f"{GP2_CLINICAL_RELEASE_PATH}/raw_genotypes/{ancestry}/{ancestry}_release10_vwb"


GP2_IMPUTED_GENO_PATH = f"{GP2_CLINICAL_RELEASE_PATH}/imputed_genotypes"

REFFLAT_HG38 = f"{BASE_DIR}/refFlat_HG38_all_chr.txt"

if not os.path.isfile(REFFLAT_HG38):
    # Download refFlat file for HG38
    ! wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz
    ! gunzip refFlat.txt.gz
    ! mv /home/jupyter/refFlat.txt $BASE_DIR/refFlat_HG38_all_chr.txt

# Covariate File

In [14]:
# Let's load the master key
key = pd.read_csv(MASTER_KEY)
if (debug):
    print("Checking size for ancestry key before removing related")
    print(f"Key size for all ancestries is: {key.shape}")
    print(key.head())
    print("Checking amp-pd")
    amppd_counts = key['amppd_wgs'].value_counts()
    print(f"Value counts for key is: {amppd_counts}")

    

# Keep only a few columns of interest
key = key[['GP2ID', 'baseline_GP2_phenotype', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'race_for_qc', 'nba_label', 'amppd_wgs']]
# And rename those columns
key.rename(columns = {'GP2ID': 'IID',
                      'baseline_GP2_phenotype':'phenotype',
                                     'biological_sex_for_qc':'SEX',
                                     'age_at_sample_collection':'AGE',
                                     'race_for_qc':'RACE',
                                     'age_of_onset':'AAO',
                                     'nba_label':'label'
                                     }, inplace = True)
if (debug):
    print(f"Key shape after selecting and renaming a few columns is: {key.shape}")
    print(key)

# Remove samples that are also present on AMP-PD dataset as we will later on use it to validate findings on NBA
key = key[key['amppd_wgs'] != 1] 
key.reset_index(drop=True)

if (debug):
    print(f"Key shape after dropping AMP-PD samples is: {key.shape}")
    print(key)
    
#Loop over all the ancestries
demographic_results = []

for ancestry in ancestries:
    print(f'Creating covar file for {ancestry}')
    
    #Define ancestry specific reference files
    RELATED_SAMPLES = f"{GP2_META_RELEASE_PATH}/related_samples/{ancestry}_release11_vwb.related"
    GP2_RAW_EIGENVEC = f"{GP2_RAW_GENO_PATH}/{ancestry}/{ancestry}_release11_vwb.eigenvec"

    ## Create folder if it doesn't exist
    WORK_DIR = f'{BASE_DIR}/{ancestry}'
    os.makedirs(WORK_DIR, exist_ok=True) 
    
    ## Subset to keep ancestry of interest
    ancestry_key = key[key['label']==ancestry].copy()
    ancestry_key.reset_index(drop=True)

    # Check ancestry key size before removing related
    if (debug):
        print("Checking size for ancestry key before removing related")
        print(f"Ancestry key size for {ancestry} is: {ancestry_key.shape}")
        ancestry_key
        
    # Load information about related individuals
    related_df = pd.read_csv(RELATED_SAMPLES)
    if (debug):
        print("Head for related_df")
        related_df.head()
        print(f"Related dataframe size for {ancestry} is: {related_df.shape}")

    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])

    # Check value counts of related and remove only one related individual
    ancestry_key = ancestry_key[~ancestry_key["IID"].isin(related_list)]

    # Check size after removing related
    if (debug):
        print("Checking size for ancestry key")
        print(f"Ancestry key size for {ancestry} is: {ancestry_key.shape}")
        ancestry_key

    # Convert phenotype to number
    # PD = 2; control = 1, Missing = -9 or 0
    pheno_mapping = {"PD": 2, "Control": 1}
    ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')

    # Check value counts of pheno
    if (debug):
        print("Checking value counts for PHENO")
        print(f"PHENO count for {ancestry} \n {ancestry_key['PHENO'].value_counts(dropna=False)}")

    # Convert sex to number (1/2/-9)
    # Female = 2; male = 1; missing = -9
    sex_mapping = {"Female": 2, "Male": 1}
    ancestry_key['SEX'] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')

    # Check value counts of SEX
    if (debug):
        print("Checking value counts for SEX")
        print(f"SEX count for {ancestry} \n {ancestry_key['SEX'].value_counts(dropna=False)}")
    
    ## Get the PCs
    pcs = pd.read_csv(GP2_RAW_EIGENVEC, sep='\t', header=0)

    selected_columns = ['IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
    
    pcs = pd.DataFrame(data=pcs.iloc[:, 1:7].values, columns=selected_columns)

    # Drop the first row (since it's now the column names)
    pcs = pcs.drop(0)

    # Reset the index to remove any potential issues
    pcs = pcs.reset_index(drop=True)

    # Display the resulting DataFrame
    if (debug):
        print("Display the PCs dataframe")
        print(pcs)

    ## Make covariate file
    df = pd.merge(pcs, ancestry_key, on='IID', how='left')
    if (debug):
        print("Display columns after merge between PCs and ancestry_key")
        df.columns


    #Make additional columns - FID, fatid and matid - these are needed for RVtests!!
    #RVtests needs the first 5 columns to be fid, iid, fatid, matid and sex otherwise it does not run correctly
    #Uppercase column name is ok
    #See https://zhanxw.github.io/rvtests/#phenotype-file
    df['FID'] = df['IID']
    df['FATID'] = 0
    df['MATID'] = 0

    ## Clean up and keep columns we need
    final_df = df[['FID','IID', 'FATID', 'MATID', 'SEX', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'AGE', 'AAO']].copy()
    if (debug):
        print("Showing final_df before droping NA") 
        final_df
        final_df.shape

    final_df=final_df.dropna(how="any", subset=['FID','IID', 'SEX', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5'])
    if (debug):
        print("Showing final_df after droping NA") 
        final_df.shape
    
    partial_results_df = final_df.groupby('PHENO')['SEX'].value_counts().reset_index()    
    n_control_male = partial_results_df[(partial_results_df['PHENO'] == 1) & (partial_results_df['SEX'] == 1)]['count'].values[0]
    n_control_female = partial_results_df[(partial_results_df['PHENO'] == 1) & (partial_results_df['SEX'] == 2)]['count'].values[0]
    n_case_male = partial_results_df[(partial_results_df['PHENO'] == 2) & (partial_results_df['SEX'] == 1)]['count'].values[0]
    n_case_female = partial_results_df[(partial_results_df['PHENO'] == 2) & (partial_results_df['SEX'] == 2)]['count'].values[0]

    n_cases = n_case_male + n_case_female
    n_control = n_control_male + n_control_female

    # Check the final count for ancestry
    if (debug):
        print(f"Final count in {ancestry} for PHENO and SEX is: \n {final_df.groupby(['PHENO'])['SEX'].value_counts()}")

    ## Make file of sample IDs to keep
    samples_toKeep = final_df[['FID', 'IID']].copy()
    samples_toKeep.to_csv(f'{WORK_DIR}/{ancestry}.samplestoKeep.txt', sep = '\t', index=False, header=None)

    ## Save your covariate file
    final_df.to_csv(f'{WORK_DIR}/{ancestry}_covariate_file.txt', sep = '\t', index=False)

    # Fill out table with demograhic results
    demographic_results.append({
        "Ancestry": ancestry,
        "Total": len(final_df.axes[0]), 
        "N_Cases": n_cases,
        "N_Controls": n_control,
        "Control_Male": n_control_male,
        "Control_Female": n_control_female,
        "Case_Male": n_case_male,
        "Case_Female": n_case_female
    })
    
    ## check to make sure file was created and saved
    #! ls {WORK_DIR}/{ancestry}_covariate_file.txt
    #! wc -l {WORK_DIR}/{ancestry}_covariate_file.txt

# Write demographis table to disk
demographics_table = pd.DataFrame(demographic_results)
demographics_table.sort_values(by='Ancestry').to_csv(f'{BASE_DIR}/demographics.csv', index=False)

Creating covar file for AMR
Creating covar file for CAS
Creating covar file for EAS
Creating covar file for AAC
Creating covar file for MDE
Creating covar file for AFR
Creating covar file for EUR
Creating covar file for AJ
Creating covar file for SAS


# Annotation using ANNOVAR

* *DNAJC13* from NCBI gene
* hg38 (chr3:132417502-132539032)

In [ ]:
variants_summary = []
rare_variants_summary = []
#Loop over all the ancestries
for ancestry in ancestries:
    ## Define variables
    WORK_DIR = f'{BASE_DIR}/{ancestry}'
    GP2_IMPUTED_FILENAME = f"{GP2_CLINICAL_RELEASE_PATH}/imputed_genotypes/{ancestry}/chr{CHNUM}_{ancestry}_release11_vwb"

    file_path = f'{WORK_DIR}/{ancestry}_DNAJC13.annovar.hg38_multianno.txt'

    print(f'\n======================================\n')
    print(f'Running annotation for ancestry: {ancestry} \n')
    if not os.path.exists(file_path):
        ## extract region using plink
        print(f'Extract region using plink for ancestry: {ancestry} \n')

        ! /home/jupyter/tools/plink2 --pfile {GP2_IMPUTED_FILENAME} \
        --chr 3 \
        --from-bp 132417502  \
        --to-bp 132539032 \
        --recode vcf id-paste=iid \
        --mac 2 \
        --out {WORK_DIR}/{ancestry}_DNAJC13

        ### Bgzip and Tabix (zip and index the file)
        ! bgzip -f {WORK_DIR}/{ancestry}_DNAJC13.vcf
        ! tabix -f -p vcf {WORK_DIR}/{ancestry}_DNAJC13.vcf.gz

        ## annotate using ANNOVAR
        ! perl /home/jupyter/tools/annovar/table_annovar.pl {WORK_DIR}/{ancestry}_DNAJC13.vcf.gz /home/jupyter/tools/annovar/humandb/ -buildver hg38 \
        -out {WORK_DIR}/{ancestry}_DNAJC13.annovar \
        -remove -protocol refGene,clinvar_20140902 \
        -operation g,f \
        --nopolish \
        -nastring . \
        -vcfinput

    gene = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.annovar.hg38_multianno.txt', sep = '\t')
    #display(gene)
    
    ## Filter intronic
    intronic = gene[(gene['Func.refGene'] == 'intronic')]
    intronic_rare = gene[(gene['Func.refGene'] == 'intronic') & (gene['Otherinfo1'] < 0.01)]

    ## Filter UTR3
    utr3 = gene[(gene['Func.refGene'] == 'UTR3')]
    utr3_rare = gene[(gene['Func.refGene'] == 'UTR3') & (gene['Otherinfo1'] < 0.01)]

    ## Filter UTR5
    utr5 = gene[(gene['Func.refGene'] == 'UTR5')]
    utr5_rare = gene[(gene['Func.refGene'] == 'UTR5') & (gene['Otherinfo1'] < 0.01)]

    ## Filter exonic and synonymous variants
    coding_synonymous = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'synonymous SNV')]
    coding_synonymous_rare = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'synonymous SNV') & (gene['Otherinfo1'] < 0.01)]

    ## Filter exonic and non-synonymous variants
    coding_nonsynonymous = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonsynonymous SNV')]
    coding_nonsynonymous_rare = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonsynonymous SNV') & (gene['Otherinfo1'] < 0.01)]

    ## Filter rare variants 
    rare_variants = gene[gene['Otherinfo1'] < 0.01]
    
    totalVariants = len(gene.axes[0])
    totalIntronic = len(intronic.axes[0])
    totalUTR3 = len(utr3.axes[0])
    totalUTR5 = len(utr5.axes[0])
    totalExonicSyn = len(coding_synonymous.axes[0])
    totalExonicNonSyn = len(coding_nonsynonymous.axes[0])

    numrarevariants = len(rare_variants.axes[0])
    rareIntronic = len(intronic_rare.axes[0])
    rareUTR3 = len(utr3_rare.axes[0])
    rareUTR5 = len(utr5_rare.axes[0])
    rareExonicSyn = len(coding_synonymous_rare.axes[0])
    rareExonicNonSyn = len(coding_nonsynonymous_rare.axes[0])

    
    print("Total Variants: ", totalVariants)
    print("Intronic:", totalIntronic)
    print("UTR3:", totalUTR3)
    print("UTR5:", totalUTR5)
    print("Exonic Syn:", totalExonicSyn)
    print("Exonic NonSyn:", totalExonicNonSyn)

    print("-------")
    print("Rare Variants: ", numrarevariants)
    print("Intronic:", rareIntronic)
    print("UTR3:", rareUTR3)
    print("UTR5:", rareUTR5)
    print("Exonic Syn:", rareExonicSyn)
    print("Exonic NonSyn:", rareExonicNonSyn)
    # Save in PLINK format

    ## For rvtests
    rare_variants_toKeep = rare_variants[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
    rare_variants_toKeep.to_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.rare.variantstoKeep.txt', sep="\t", index=False, header=False)

    ## For association analysis   
    variants_toKeep = coding_nonsynonymous[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
    variants_toKeep.to_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.variantstoKeep.txt', sep="\t", index=False, header=False)
    #variants_toKeep

    ## check to make sure file was created and saved
    #! ls {WORK_DIR}

    ## Write variants summary
    variants_summary.append({
        "Ancestry": ancestry,
        "Total": totalVariants, 
        "Intronic": totalIntronic, 
        "UTR3": totalUTR3,
        "UTR5": totalUTR5,
        "ExonicSyn": totalExonicSyn,
        "ExonicNonSyn": totalExonicNonSyn})

    ## Write rare variants summary
    rare_variants_summary.append({
        "Ancestry": ancestry,
        "Total": numrarevariants, 
        "Intronic": rareIntronic, 
        "UTR3": rareUTR3,
        "UTR5": rareUTR5,
        "ExonicSyn": rareExonicSyn,
        "ExonicNonSyn": rareExonicNonSyn})

# Write variants table to disk
variants_table = pd.DataFrame(variants_summary)
variants_table.sort_values(by='Ancestry').to_csv(f'{BASE_DIR}/total_variants.csv', index=False)

# Write variants table to disk
rare_variants_table = pd.DataFrame(rare_variants_summary)
rare_variants_table.sort_values(by='Ancestry').to_csv(f'{BASE_DIR}/rare_variants.csv', index=False)



Running annotation for ancestry: AMR 

Total Variants:  1338
Intronic: 1281
UTR3: 11
UTR5: 2
Exonic Syn: 15
Exonic NonSyn: 29
-------
Rare Variants:  1089
Intronic: 1039
UTR3: 9
UTR5: 2
Exonic Syn: 14
Exonic NonSyn: 25


Running annotation for ancestry: CAS 

Total Variants:  961
Intronic: 928
UTR3: 8
UTR5: 3
Exonic Syn: 6
Exonic NonSyn: 16
-------
Rare Variants:  740
Intronic: 714
UTR3: 6
UTR5: 3
Exonic Syn: 5
Exonic NonSyn: 12


Running annotation for ancestry: EAS 

Total Variants:  1376
Intronic: 1333
UTR3: 6
UTR5: 6
Exonic Syn: 10
Exonic NonSyn: 21
-------
Rare Variants:  1153
Intronic: 1113
UTR3: 4
UTR5: 6
Exonic Syn: 10
Exonic NonSyn: 20


Running annotation for ancestry: AAC 

Total Variants:  1199
Intronic: 1150
UTR3: 12
UTR5: 3
Exonic Syn: 14
Exonic NonSyn: 20
-------
Rare Variants:  693
Intronic: 657
UTR3: 8
UTR5: 2
Exonic Syn: 11
Exonic NonSyn: 15


Running annotation for ancestry: MDE 

Total Variants:  1008
Intronic: 971
UTR3: 8
UTR5: 2
Exonic Syn: 9
Exonic NonSyn: 18
-

# Burden analysis using RVTest

In [5]:
#Loop over all the ancestries
for ancestry in ancestries:
    WORK_DIR = f'{BASE_DIR}/{ancestry}'
    GP2_IMPUTED_FILENAME = f"{GP2_CLINICAL_RELEASE_PATH}/imputed_genotypes/{ancestry}/chr{CHNUM}_{ancestry}_release11_vwb"
        
    # Prepare the file format for RVTESTs
    ## Extract relevant variants

    file_path = f'{WORK_DIR}/{ancestry}_DNAJC13.rare.variants.vcf.gz'
    if not os.path.exists(file_path):
        print(f'Running plink to extract rare variants for ancestry: {ancestry}')
        
        ! /home/jupyter/tools/plink2 \
        --pfile {GP2_IMPUTED_FILENAME} \
        --keep {WORK_DIR}/{ancestry}.samplestoKeep.txt \
        --extract range {WORK_DIR}/{ancestry}_DNAJC13.rare.variantstoKeep.txt \
        --recode vcf-iid \
        --out {WORK_DIR}/{ancestry}_DNAJC13.rare_variants
        
        # Print the command to be executed (for debugging purposes)
        print(f'Running bgzip and tabix for rare variants for ancestry: {ancestry}')
        
        ## Bgzip and Tabix (zip and index the file)
        ! bgzip -f {WORK_DIR}/{ancestry}_DNAJC13.rare_variants.vcf
        ! tabix -f -p vcf {WORK_DIR}/{ancestry}_DNAJC13.rare_variants.vcf.gz

    ## RVtests with covariates 
    #Make sure the pheno and covariate file starts with the first 5 columsn: fid, iid, fatid, matid, sex
    #The pheno-name flag only works when the pheno/covar file is structured properly
    
    file_path = f'{WORK_DIR}/{ancestry}_DNAJC13.burden.rare_variants.Skat.assoc'
    if not os.path.exists(file_path):
        # Print the command to be executed (for debugging purposes)
        print(f'Running RVtests for rare variants for ancestry: {ancestry}')
        
        ! /home/jupyter/tools/rvtests/executable/rvtest --noweb --hide-covar \
        --kernel skat,skato \
        --inVcf {WORK_DIR}/{ancestry}_DNAJC13.rare_variants.vcf.gz \
        --pheno {WORK_DIR}/{ancestry}_covariate_file.txt \
        --pheno-name PHENO \
        --gene DNAJC13 \
        --geneFile {BASE_DIR}/refFlat_HG38_all_chr.txt \
        --covar {WORK_DIR}/{ancestry}_covariate_file.txt \
        --covar-name SEX,AGE,PC1,PC2,PC3,PC4,PC5 \
        --out {WORK_DIR}/{ancestry}_DNAJC13.burden.rare_variants 

Running plink to extract rare variants for ancestry: EAS
PLINK v2.0.0-a.6.9LM AVX2 AMD (29 Jan 2025)        cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/202512_R11_Batch/EAS/EAS_DNAJC13.rare_variants.log.
Options in effect:
  --export vcf-iid
  --extract range /home/jupyter/workspace/ws_files/202512_R11_Batch/EAS/EAS_DNAJC13.rare.variantstoKeep.txt
  --keep /home/jupyter/workspace/ws_files/202512_R11_Batch/EAS/EAS.samplestoKeep.txt
  --out /home/jupyter/workspace/ws_files/202512_R11_Batch/EAS/EAS_DNAJC13.rare_variants
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/EAS/chr3_EAS_release11_vwb

Start time: Thu Dec  4 13:52:07 2025
Note: --export 'vcf-iid' modifier is deprecated.  Use 'vcf' + 'id-paste=iid'.
26041 MiB RAM detected, ~24880 available; reserving 13020 MiB for main
workspace.
Using up to 4 compute threads.
7965 samples (3328 females, 4637 males;

In [6]:
# Create a summary table
burden_results = []

for ancestry in ancestries:
    WORK_DIR = f'{BASE_DIR}/{ancestry}'
    skat = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.burden.rare_variants.Skat.assoc', sep='\s+')
    skatO = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.burden.rare_variants.SkatO.assoc', sep='\s+')

    burden_results.append({
        'Ancestry': ancestry,
        'N_var': skat.at[0, 'NumVar'],
        'Skat_pval': skat.at[0, 'Pvalue'],
        'SkatO_pval': skatO.at[0,'Pvalue']
    })

burden_table = pd.DataFrame(burden_results)
burden_table.sort_values(by='Ancestry').to_csv(f'{BASE_DIR}/burden_table.csv', index=False)

# Case/Control Frequencies

In [ ]:
#Loop over all the ancestries

for ancestry in ancestries:
    WORK_DIR = f'{BASE_DIR}/{ancestry}'
    GP2_IMPUTED_FILENAME = f"{GP2_CLINICAL_RELEASE_PATH}/imputed_genotypes/{ancestry}/chr{CHNUM}_{ancestry}_release11_vwb"
                
    # Print the command to be executed (for debugging purposes)
    print(f'Running plink to extract common variants for ancestry: {ancestry} \n')

    #Prepare the file format for PLINK1.9
    ## extract common coding nonsyn variants
    file_path=f'{WORK_DIR}/{ancestry}_DNAJC13.common.variants.bed'
    if not os.path.exists(file_path):
        ! /home/jupyter/tools/plink2 --pfile {GP2_IMPUTED_FILENAME} \
        --make-bed \
        --keep {WORK_DIR}/{ancestry}.samplestoKeep.txt \
        --extract range {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.variantstoKeep.txt \
        --pheno {WORK_DIR}/{ancestry}_covariate_file.txt \
        --pheno-col-nums 6 \
        --update-sex {WORK_DIR}/{ancestry}_covariate_file.txt 'col-num='5 \
        --maf 0.01 \
        --out {WORK_DIR}/{ancestry}_DNAJC13.common.variants

    ##Perform basic association analysis
    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/{ancestry}_DNAJC13.common.variants \
    --assoc \
    --a2-allele {BASE_DIR}/ref_allele_list.txt --real-ref-alleles \
    --out {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn --ci 0.95

    # Read association results into dataframe
    # ! cat {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.assoc
    assoc = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.assoc', sep='\s+', usecols=['SNP', 'A1', 'A2', 'F_A', 'F_U', 'P', 'OR', 'L95', 'U95'])
    # assoc

    ## repeat association study applying multiple testing corrections for the raw p-values (like Bonferroni post-hoc)
    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/{ancestry}_DNAJC13.common.variants \
    --assoc \
    --a2-allele {BASE_DIR}/ref_allele_list.txt --real-ref-alleles \
    --out {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn --adjust

    # Read adjusted association results into dataframe
    #! cat {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.assoc.adjusted
    assoc_adjusted = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.assoc.adjusted', sep='\s+', usecols=['SNP', 'BONF'])
    #assoc_adjusted

    # Calculate frequencies
    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/{ancestry}_DNAJC13.common.variants \
    --freq \
    --a2-allele {BASE_DIR}/ref_allele_list.txt --real-ref-alleles \
    --out {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn

    # Read frequencies results into dataframe
    # ! cat {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.frq
    freq = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.frq', sep='\s+', usecols=['SNP', 'MAF'])
    # freq.head()

    # Calculate frequencies Case/Control
    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/{ancestry}_DNAJC13.common.variants \
    --freq case-control \
    --a2-allele {BASE_DIR}/ref_allele_list.txt --real-ref-alleles \
    --out {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn

    # Read frequencies case/control results into dataframe
    # ! cat {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.frq.cc
    freq_cc = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.frq.cc', sep='\s+', usecols=['SNP', 'MAF_A', 'MAF_U'])
    # freq_cc.head()

    # Merge frequencies
    all_freq = pd.merge(freq, freq_cc, on="SNP", how="left")
    # all_freq

    # Merge associations and frequencies
    assoc_and_freq = pd.merge(all_freq, assoc, on="SNP", how="left")
    # assoc_and_freq
    assoc_and_freq = pd.merge(assoc_and_freq, assoc_adjusted, on="SNP", how="left")
    # assoc_and_freq.head()

    # Recode files for allele counting
    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/{ancestry}_DNAJC13.common.variants \
    --recode A \
    --a2-allele {BASE_DIR}/ref_allele_list.txt --real-ref-alleles \
    --out {WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn

    # Read recode into dataframe
    recode = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.raw', sep='\s+')
    # recode.head()

    # Make a list from the column names
    column_names = recode.columns.tolist()

    # Drop the first 6 columns to keep the variants
    variants = column_names[6:]

    print(f'Number of variants in {ancestry} for DNAJC13: {len(variants)}')
    variants

    # Pre-filter the dataset
    cases_data = recode[recode['PHENOTYPE'] == 2]
    controls_data = recode[recode['PHENOTYPE'] == 1]

    results = []

    for variant in variants:
        # For cases
        hom_cases = cases_data[cases_data[variant] == 2].shape[0]
        het_cases = cases_data[cases_data[variant] == 1].shape[0]
        total_cases = cases_data.shape[0]
        # freq_cases = (hom_cases + het_cases) / total_cases

        # For controls
        hom_controls = controls_data[controls_data[variant] == 2].shape[0]
        het_controls = controls_data[controls_data[variant] == 1].shape[0]
        total_controls = controls_data.shape[0]
        # freq_controls = (hom_controls + het_controls) / total_controls

        results.append({
            'Variant': variant,
            'Hom_Cases': hom_cases,
            'Het_Cases': het_cases,
            'Total_Cases': total_cases,
            # 'Freq in Cases': freq_cases,
            'Hom_Controls': hom_controls,
            'Het_Controls': het_controls,
            'Total_Controls': total_controls,
            # 'Freq in Controls': freq_controls
        })

    # Return
    df_results = pd.DataFrame(results)
    df_results['SNP'] = df_results['Variant'].apply(lambda x: x.rsplit('_', 1)[0])
    #df_results

    ## Merge with assoc results
    full_results = pd.merge(assoc_and_freq, df_results, on="SNP", how="left")

    clean_full_results = full_results[['SNP', 'A1', 'A2', 'F_A', 'F_U', 'P', 'OR','L95', 'U95', 'BONF',
                                       'MAF', 'MAF_A', 'MAF_U',
                                       'Hom_Cases', 'Het_Cases', 'Total_Cases',
                                       'Hom_Controls','Het_Controls', 'Total_Controls']].copy()

    print(clean_full_results.shape)
    print(f'No. of SNPs: {clean_full_results.shape[0]}')
    clean_full_results

    # Look at significant SNPs, if any
    sig_freq = clean_full_results[clean_full_results['P']<0.05]
    sig_snps = sig_freq['SNP'].tolist()
    sig_snps

    # Look at significant SNPs, if any
    sig_df_results = clean_full_results[clean_full_results['SNP'].isin(sig_snps)]
    sig_df_results

    for snp in sig_snps:
        ! grep {snp} {GP2_IMPUTED_FILENAME}.pvar

    # Save files to VM
    clean_full_results.to_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.fullVariantInformation.txt', sep="\t", index=False)
    sig_df_results.to_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.SignificantVariantInformation.txt' , sep="\t", index=False)

In [8]:
# Make summary file for table 1
assoc_results = []

for ancestry in ancestries:
    WORK_DIR = f'{BASE_DIR}/{ancestry}'
    GP2_IMPUTED_FILENAME = f"{GP2_CLINICAL_RELEASE_PATH}/imputed_genotypes/{ancestry}/chr{CHNUM}_{ancestry}_release11_vwb"

    print(f'\nAnalysis of assoc results for {ancestry}')
    
    #load significant variants
    sig_var = pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.SignificantVariantInformation.txt', sep='\t')

    if sig_var.empty:
        print(f'No significant variants were found in {ancestry}')
    else:
        num_rows = len(sig_var)
        print(f'We found {num_rows} significant variants')

        coding_nonsyn=pd.read_csv(f'{WORK_DIR}/{ancestry}_DNAJC13.coding_nonsyn.raw', sep="\s+")
        final_df=pd.read_csv(f'{WORK_DIR}/{ancestry}_covariate_file.txt', sep="\t")
              
        full_Table = pd.merge(final_df, coding_nonsyn, on = "IID", how = "inner")

        for index, row in sig_var.iterrows():

            cust_var = row['SNP'] + "_" + row['A1']

            #extract the raw data for the variant of interest
            clear_table = full_Table[["IID", "AGE", "AAO", "PHENO", cust_var]]
            table = clear_table[clear_table[cust_var]>0]

            ! grep {row['SNP']} {GP2_IMPUTED_FILENAME}.pvar
            
            assoc_results.append({
                'Ancestry': ancestry,
                'Location': row['SNP'],
                'Freq_cases': row['F_A'],
                'Mean_Age_cases': table[table['PHENO'] == 2]['AGE'].mean(),
                'Std_Age_cases': table[table['PHENO'] == 2]['AGE'].std(),
                'Num_cases': row['Hom_Cases'] + row['Het_Cases'],
                'Mean_AgeAtOnset_cases': table[table['PHENO'] == 2]['AAO'].mean(),
                'Std_AgeAtOnset_cases': table[table['PHENO'] == 2]['AAO'].std(),
                'Freq_controls': row['F_U'],
                'Mean_Age_controls': table[table['PHENO'] == 1]['AGE'].mean(),
                'Std_Age_controls': table[table['PHENO'] == 1]['AGE'].std(),
                'Num_controls': row['Hom_Controls'] + row['Het_Controls'],
                'OR': row['OR'],
                'L95': row['L95'],
                'U95': row['U95'],
                'pval': row['P'],
                'Bonf': row['BONF'],
            })

assoc_table = pd.DataFrame(assoc_results)
assoc_table.sort_values(by=['Location', 'Ancestry']).to_csv(f'{BASE_DIR}/assoc_table.csv', index=False)


Analysis of assoc results for EAS
We found 1 significant variants
3	132499779	chr3:132499779:G:T	G	T	IMPUTED;AF=0.821179;MAF=0.178821;AVG_CS=0.921089;R2=0.613542

Analysis of assoc results for CAS
No significant variants were found in CAS

Analysis of assoc results for AJ
No significant variants were found in AJ

Analysis of assoc results for EUR
We found 2 significant variants
3	132499779	chr3:132499779:G:T	G	T	PASS	IMPUTED;MAF=0.468366;AVG_CS=0.997401;R2=0.990361;AF=0.465693
3	132502295	chr3:132502295:C:T	C	T	PASS	TYPED;IMPUTED;MAF=0.0292983;AVG_CS=0.999946;R2=0.998425;ER2=0.966604;AF=0.0293241

Analysis of assoc results for MDE
We found 1 significant variants
3	132502295	chr3:132502295:C:T	C	T	TYPED;IMPUTED;AF=0.0349398;MAF=0.0349398;AVG_CS=0.999835;R2=0.995642;ER2=0.926569

Analysis of assoc results for SAS
We found 1 significant variants
3	132499777	chr3:132499777:G:A	G	A	TYPED;IMPUTED;AF=0.014845;MAF=0.014845;AVG_CS=0.999884;R2=0.992418;ER2=0.875143

Analysis of assoc results fo